# DataFrame joins in spark
A spark application bring in a large number of dataFrames
- Joins are all about bringing two dataFrames togeather and these two dataFrames are termed as left dataFrame and right dataFrame.
- We combing the left dataFrame with the right dataFrame using two things.
    - Join Condition/Expression
    - Join type
- When performing joins we always start with the left dataFrame and pass in the right dataFrame as the first argument of the join method. The join method takes in two more argument join expression and the join method.
    - ```order_df.join(product_df,join_expr,join_type)```
- Inner join is the default value of the join_type argument
- How do spark handles join internally? 
    - Spark is going to take first row from your left dataFrame and evaluate the join_expression for all the rows in the right dataFrame to find a match
    - After this as the next step Spark combines these matching records from the left and the right dataFrames to create a new dataFrame that is where the join type comes into the picture.
- There is one thing that you need to make sure you avoid and that is column abiguity 
    - You can avoid column ambiguity by re-naming those columns that are same in both the left and right dataFrames. even before performing the join operations on them.
    - Example ; 
    main.py file
    ```python
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing inner join opertion on the two dataFrames
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    inner_join_df = df_joins.inner_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    inner_join_df = inner_join_df.select("order_id","prod_id","unit_price","qty","prod_name","list_price","reorder_qty")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=inner_join_df,spark_df_name="inner_join_df")
    ```
    joins.py file
    ```python
    from lib.logger import Log4j,LogSparkDataframe
    from lib.app_monitor import GetDataFrameMemory
    class DataFrameJoins:
        def __init__(self,spark):
            self.spark = spark
            self.logger = Log4j(spark)
            self.mem = GetDataFrameMemory(spark)
            self.metrics = LogSparkDataframe(spark)
        def inner_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"inner")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after inner join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # left join is also called left outer join
        def left_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"left")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after left join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # right join is also called right outer join
        def right_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"right")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after right join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # here outer join is actually full outer join there is no difference between them
        def outer_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"outer")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after outer join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
    ```
## Here is the Final code for joins
### lib
#### app_monitor.py
```python
from pyspark import StorageLevel
from .logger import Log4j

class GetDataFrameMemory:
    """
    This class measures the approximate memory usage of a Spark DataFrame.
    It uses caching and executor memory metrics to get insights.
    """
    def __init__(self, spark):
        self.spark_session_object = spark
        self.logger = Log4j(spark)

    def get_mem_usage(self, spark_df):
        try:
            # Persist DataFrame in memory so Spark tracks it
            spark_df.persist(StorageLevel.MEMORY_ONLY)
            spark_df.count()  # Trigger cache materialization

            # Get JVM reference
            jvm = self.spark_session_object._jvm

            # Get executor memory status (Scala Map)
            scala_map = self.spark_session_object._jsc.sc().getExecutorMemoryStatus()

            # Convert Scala Map -> Java Map
            java_map = jvm.scala.collection.JavaConverters.mapAsJavaMapConverter(scala_map).asJava()

            # Convert Java Map -> Python dict
            memory_info = {
                entry.getKey(): (entry.getValue()._1(), entry.getValue()._2())
                for entry in java_map.entrySet()
            }

            if not memory_info:
                self.logger.warn("No executor memory metrics found — possibly running in local mode.")
                approx_size = spark_df.rdd.map(lambda x: len(str(x))).sum()
                self.logger.debug(f"Approximate DataFrame size (bytes) [Driver]: {self.convert_bytes_to_mb(approx_size)} MB")
                return {"mem": approx_size}

            self.logger.debug(f"Executor memory usage snapshot: {self.convert_bytes_to_mb(memory_info)} MB")

            # Compute used memory per executor
            mem_used = {
                host: total - remaining
                for host, (total, remaining) in memory_info.items()
            }

            self.logger.debug(f"Memory used by executors (MB): {self.convert_bytes_to_mb(mem_used)} MB")
            return {"mem": self.convert_bytes_to_mb(mem_used)}

        except Exception as e:
            self.logger.error(f"Error while fetching DataFrame memory usage: {e}")
            return {}
    def convert_bytes_to_mb(self, value):
        try:
            if isinstance(value, (int, float)):
                return round(value / (1024 * 1024), 2)
            elif isinstance(value, tuple):
                return tuple(round(v / (1024 * 1024), 2) if isinstance(v, (int, float)) else v for v in value)
            elif isinstance(value, dict):
                return {k: self.convert_bytes_to_mb(v) for k, v in value.items()}
            elif isinstance(value, (list, set)):
                return [self.convert_bytes_to_mb(v) for v in value]
            else:
                return value
        except Exception as e:
            self.logger.error(f"Error in convert_bytes_to_mb: {e}")
            return value
```
#### cleanup_up_file_system.py
```python
import os
import shutil

import time

"""
This class will cleanup the data when the spark application re-runs

The files like logs, metastore , spark-warehouse etc.. will be cleaned up (deleted from the file system)
"""
class CleanupAppFileSystemOnReRun:
    def __init__(self,project_dir):
        self.project_dir = project_dir

    def execute_cleanup(self,clean_logs:bool = False):
        self.derby_logs_cleanup()
        self.spark_warehouse_cleanup()
        self.metastore_cleanup()
        if clean_logs == True:
            self.logs_cleanup()
        # time.sleep(5)

    """This will cleanup the derby.logs"""
    def derby_logs_cleanup(self):
        try:
            derby_logs_dir = os.path.join(self.project_dir, "derby.log")
            if os.path.exists(derby_logs_dir):
                os.remove(derby_logs_dir)
                print(f"Deleted existing derby.log file: {derby_logs_dir}")
            else:
                print(f"derby.log file does not exists: {derby_logs_dir}")
        except Exception as e:
            print(str(e))
            raise
    
    """This will cleanup the spark_warehouse"""
    def spark_warehouse_cleanup(self):
        try:
            spark_warehouse_dir = os.path.join(self.project_dir, "spark-warehouse")
            if os.path.exists(spark_warehouse_dir):
                shutil.rmtree(spark_warehouse_dir)
                print(f"Deleted existing spark-warehouse directory: {spark_warehouse_dir}")
            else:
                print(f"spark-warehouse directory does not exists: {spark_warehouse_dir}")
        except Exception as e:
            print(str(e))
            raise

    """This will cleanup the metastore_db"""
    def metastore_cleanup(self):
        try:
            metastore_dir = os.path.join(self.project_dir, "metastore_db")
            if os.path.exists(metastore_dir):
                shutil.rmtree(metastore_dir)
                print(f"Deleted existing metastore directory: {metastore_dir}")
            else:
                print(f"metastore directory does not exists: {metastore_dir}")
        except Exception as e:
            print(str(e))
            raise

    """This will clean the logs folder"""
    def logs_cleanup(self):
        try:
            log_dir = os.path.join(self.project_dir, "log4j_properties", "logs")
            # delete the log directory
            if os.path.exists(log_dir):
                shutil.rmtree(log_dir)
                print(f"Deleted existing log directory: {log_dir}")
            else:
                print(f"Log directory does not exist: {log_dir}")
        except Exception as e:
            print(str(e))
            raise
```
#### logger.py
```python
class Log4j:
    def __init__(self, spark):
        # Get a log4j instance
        log4j = spark._jvm.org.apache.log4j
        # Create a logger attribute
        # put your organization name as a root class 
        root_class = "credencys.aditya.spark"
        conf = spark.sparkContext.getConf()
        app_name = conf.get("spark.app.name")
        self.logger = log4j.LogManager.getLogger(root_class + "." + app_name)

    def warn(self,message):
        self.logger.warn(message)
    
    def info(self,message):
        self.logger.info(message)
    
    def error(self, message):
        self.logger.error(message)
    
    def debug(self,message):
        self.logger.debug(message)

class LogSparkDataframe:
    def __init__(self,spark):
        self.sp = spark
        self.logger = Log4j(spark)
    # This will log the dataframe
    def log_df(self,spark_df,spark_df_name):
        self.logger.info(f">>>>> {spark_df_name} dataframe:\n{spark_df.limit(25).toPandas().to_string(index=False)}")
    # This will log the database schema
    def log_df_metrics(self,spark_df,spark_df_name):
        schema_str = spark_df._jdf.schema().treeString()
        self.logger.debug(f"{spark_df_name} :: operation - LogSparkDataframe :: Spark DataFrame Schema (expanded): {schema_str}")
```
#### utils.py
```python
import configparser
from pyspark import SparkConf
import os


"""
This function will load the configuration from spark.conf file and return a spark conf object
"""
def get_spark_app_config():
    spark_conf = SparkConf()
    config = configparser.ConfigParser()
    # Read the spark.conf file from the dirtectory
    config.read(os.path.join(os.getcwd(),"spark.conf"))

    # Loop through the configs and set it to the spark conf
    for (key, val) in config.items("SPARK_APP_CONFIGS"):
        spark_conf.set(key,val)
    return spark_conf
```
#### write_df.py
```python
from lib.logger import Log4j
class ExportSparkDataFrame:
    def __init__(self,spark):
        self.logger = Log4j(spark)
    def export_df_parquet(self,spark_df,save_mode: str = "overwrite",output_path : str="",max_rec : int = 100):
        try:
            (
                spark_df.write
                .format("parquet")
                .mode(save_mode)
                .option("path",output_path)
                .option("maxRecorsdPerFile",max_rec)
                .save()
            )
            return True
        except Exception as e:
            self.logger.error(str(e))
            raise
    
    def export_df_avro(self,spark_df,save_mode: str = "overwrite", output_path: str = ""):
        try:
            (
                spark_df.write
                .format("avro")
                .mode(save_mode)
                .option("path",output_path)
                .save()
            )
            return True
        except Exception as e:
            self.logger.error(str(e))
            raise

    def export_df_json(self,spark_df,save_mode : str = "overwrite", output_path : str = "",column_list : list = []):
        try:
            (
                spark_df.write
                .format("json")
                .mode(save_mode)
                .option("path",output_path)
                .partitionBy(column_list[0],column_list[1])
                .save()
            )
        except Exception as e:
            self.logger.error(str(e))
            raise
```
### log4j_properties/log4j.properties
```bash
# Root logger configuration
log4j.rootCategory=INFO, console

# Console output
log4j.appender.console=org.apache.log4j.ConsoleAppender
log4j.appender.console.layout=org.apache.log4j.PatternLayout
log4j.appender.console.layout.ConversionPattern=%d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n

# ======================================================
# INFO -> logs/info.log
# ======================================================
log4j.appender.infoFile=org.apache.log4j.FileAppender
log4j.appender.infoFile.File=${custom.log.dir}/info.log
log4j.appender.infoFile.Append=true
log4j.appender.infoFile.Threshold=INFO
log4j.appender.infoFile.layout=org.apache.log4j.PatternLayout
log4j.appender.infoFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
# filter section: This will exclude warn and error logs only info logs will be appended
log4j.appender.infoFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
log4j.appender.infoFile.filter.a.LevelToMatch=INFO
log4j.appender.infoFile.filter.a.AcceptOnMatch=true
log4j.appender.infoFile.filter.b=org.apache.log4j.varia.DenyAllFilter

# WARN + ERROR -> logs/warn-error.log
# This will allow me to use the logger.py file
log4j.appender.warnErrorFile=org.apache.log4j.FileAppender
log4j.appender.warnErrorFile.File=${custom.log.dir}/warn-error.log
log4j.appender.warnErrorFile.Append=true
log4j.appender.warnErrorFile.Threshold=WARN
log4j.appender.warnErrorFile.layout=org.apache.log4j.PatternLayout
log4j.appender.warnErrorFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n

# DEBUG -> logs/debug.log
log4j.appender.debugFile=org.apache.log4j.FileAppender
log4j.appender.debugFile.File=${custom.log.dir}/debug.log
log4j.appender.debugFile.Append=true
log4j.appender.debugFile.Threshold=DEBUG
log4j.appender.debugFile.layout=org.apache.log4j.PatternLayout
log4j.appender.debugFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
# Filter: only DEBUG
log4j.appender.debugFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
log4j.appender.debugFile.filter.a.LevelToMatch=DEBUG
log4j.appender.debugFile.filter.a.AcceptOnMatch=true
log4j.appender.debugFile.filter.b=org.apache.log4j.varia.DenyAllFilter

# Loggers
# log4j.logger.org.apache.spark=DEBUG, debugFile, warnErrorFile
# log4j.additivity.org.apache.spark=false

# log4j.logger.org.apache=INFO, warnErrorFile
# log4j.additivity.org.apache=false

# Here I am defining a common root logger definitions for appenders
log4j.logger.credencys.aditya.spark=DEBUG, debugFile, warnErrorFile, infoFile
log4j.additivity.credencys.aditya.spark=false
```
### unit_testing/generate_dataframe.py
```python
from typing import List, Tuple, Any
from lib.logger import Log4j, LogSparkDataframe
from lib.app_monitor import GetDataFrameMemory
from pyspark.sql import functions as F

class GenerateDataFrame:
    def __init__(self, spark):
        self.spark = spark
        self.logger = Log4j(spark)
        self.sp_df_logger = LogSparkDataframe(spark)
        self.app_metrics = GetDataFrameMemory(spark)

    def generate_dataframe(self, data_list : List[Tuple[Any, ...]] = None,column_name_list : List[str]=None):
        self.logger.debug(f"checking for the supplied data_list: ")
        if not data_list:
            self.logger.error("DataList required to generate a dataFrame!")
            raise ValueError(f"DataList required to generate a dataFrame!")
        if not column_name_list:
            self.logger.error("Column List required to generate a dataFrame!")
            raise ValueError(f"Column List required to generate a dataFrame!")
        self.logger.debug(f"supplied data_list found : {data_list}")
        self.logger.debug(f"column name list foind {column_name_list}")

        # check if the spark session master is set to local if yes then implement repartition if not then don't
        if self.spark.sparkContext.master == "local[3]":
            generated_df = (
                self.spark
                .createDataFrame(data_list)
                .toDF(*column_name_list)
                .repartition(3)
            )
        else:
            generated_df = (
                self.spark
                .createDataFrame(data_list)
                .toDF(*column_name_list)
            )
        self.app_metrics.get_mem_usage(generated_df)
        self.sp_df_logger.log_df_metrics(spark_df=generated_df,spark_df_name="generated_df")
        return generated_df
```
### spark.conf
```bash
[SPARK_APP_CONFIGS]
saprk.app.name = SparkSqlTableDemo
spark.master = local[3]

# Setting up the dataset file name that are used in this application
file_name_csv = flight-time.csv
file_name_json = flight-time.json
file_name_parquet = flight-time.parquet
file_name_text = apache_logs.txt

# Added a shuffle sort partitions to control the no of partitions of the spark dataFrame in the spark applicaiton
spark.sql.shuffle.partitions = 2

# Tell spark to save the created table in this database
db_name = airline_db
flight_table_name = flight_data
```
### joins/df_joins.py
```python
from lib.logger import Log4j,LogSparkDataframe
from lib.app_monitor import GetDataFrameMemory
class DataFrameJoins:
    def __init__(self,spark):
        self.spark = spark
        self.logger = Log4j(spark)
        self.mem = GetDataFrameMemory(spark)
        self.metrics = LogSparkDataframe(spark)
    def inner_join_df(self,left_df,right_df,join_expression):
        try:
            # perform join operation
            result_df = left_df.join(right_df,join_expression,"inner")
            # logging memory taken by the df
            self.mem.get_mem_usage(result_df)
            # logging spark schema 
            self.logger.debug("logging intermediate dataframe after inner join operation")
            self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
            return result_df
        except Exception as e:
            self.logger.error(str(e))
            raise
    # left join is also called left outer join
    def left_join_df(self,left_df,right_df,join_expression):
        try:
            # perform join operation
            result_df = left_df.join(right_df,join_expression,"left")
            # logging memory taken by the df
            self.mem.get_mem_usage(result_df)
            # logging spark schema 
            self.logger.debug("logging intermediate dataframe after left join operation")
            self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
            return result_df
        except Exception as e:
            self.logger.error(str(e))
            raise
    # right join is also called right outer join
    def right_join_df(self,left_df,right_df,join_expression):
        try:
            # perform join operation
            result_df = left_df.join(right_df,join_expression,"right")
            # logging memory taken by the df
            self.mem.get_mem_usage(result_df)
            # logging spark schema 
            self.logger.debug("logging intermediate dataframe after right join operation")
            self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
            return result_df
        except Exception as e:
            self.logger.error(str(e))
            raise
    # here outer join is actually full outer join there is no difference between them
    def outer_join_df(self,left_df,right_df,join_expression):
        try:
            # perform join operation
            result_df = left_df.join(right_df,join_expression,"outer")
            # logging memory taken by the df
            self.mem.get_mem_usage(result_df)
            # logging spark schema 
            self.logger.debug("logging intermediate dataframe after outer join operation")
            self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
            return result_df
        except Exception as e:
            self.logger.error(str(e))
            raise
```
### main.py file
```python
from pyspark.sql import SparkSession
# import related to logging
from lib.logger import Log4j, LogSparkDataframe
# import related to custom spark configurations
from lib.utils import get_spark_app_config
# imports related to exporting dataframe
from lib.write_df import ExportSparkDataFrame
# import writing sparkdf to tables related stuff
from lib.load_df_data_into_table import LoadSparkDFIntoTable
# logging related imports 
import os

# Imports related to ingest data
from lib.ingest_data import IngestData
# Transform data
from transformations.dataframe_transformations import DataFrameTransformations
# imports related to dataframe joins
from joins.df_joins import DataFrameJoins

# imports related to cleanup when the main_app.py is re-run
from lib.clean_up_file_system import CleanupAppFileSystemOnReRun

# imports related to generating dataFrame
from unit_testing.generate_dataframe import GenerateDataFrame

if __name__ == "__main__":
    # logging related logic
    # Get the current project's directory
    project_dir = os.path.dirname(os.path.abspath(__file__))
    # cleanup loggic on main_app.py re-run
    # initialize the cleanup class
    cleanup = CleanupAppFileSystemOnReRun(project_dir)
    cleanup.execute_cleanup(clean_logs=True)

    # Get the Log4j.properties file directory
    log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
    # Save the directory where the generated log files must reside
    log_dir = os.path.join(project_dir, "log4j_properties", "logs")
    # Create the directory where the log files must be kept if not present
    os.makedirs(log_dir, exist_ok=True)

    conf = get_spark_app_config()
    spark = (
        SparkSession
        .builder
        .config(conf=conf)
        .config("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")
        .enableHiveSupport()
        .getOrCreate()
    )

    # initialize logger class 
    logger = Log4j(spark)

    # initialize the spark dataframe logger 
    sp_df_logger = LogSparkDataframe(spark)

    # logging some debug related stuff 
    logger.debug(f"log4j.properties file dir = {log4j_config_path}")
    logger.debug(f"log files dir = {log_dir}")
    logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
    
    logger.info("Reading the data from the directory")
    dataset_dir = os.path.join(project_dir,"dataset")

    ######################################
    # CREATE A DATAFRAME AND PERFORM DATAFRAME JOIN ON IT STARTS
    ######################################
    """Create DataFrame STARTS"""
    gen_df = GenerateDataFrame(spark)

    # Generate orders dataFrame
    data_list = [
                ("01", "02", 350, 1),
                ("01", "04", 580, 1),
                ("01", "07", 320, 2),
                ("02", "03", 450, 1),
                ("02", "06", 220, 1),
                ("03", "01", 195, 1),
                ("04", "09", 270, 3),
                ("04", "08", 410, 2),
                ("05", "02", 350, 1)
            ]
    column_name_list = ["order_id", "prod_id", "unit_price", "qty"]
    generated_order_df = gen_df.generate_dataframe(data_list=data_list,column_name_list=column_name_list)
    # logging dataFrame
    sp_df_logger.log_df(spark_df=generated_order_df,spark_df_name="generated_order_df")

    # Generate product_list dataFrame
    data_list = [
                    ("01", "Scroll Mouse", 250, 20),
                    ("02", "Optical Mouse", 350, 20),
                    ("03", "Wireless Mouse", 450, 50),
                    ("04", "Wireless Keyboard", 580, 50),
                    ("05", "Standard Keyboard", 360, 10),
                    ("06", "16 GB Flash Storage", 240, 100),
                    ("07", "32 GB Flash Storage", 320, 50),
                    ("08", "64 GB Flash Storage", 430, 25)
                ]
    column_name_list = ["prod_id", "prod_name", "list_price", "qty"]
    generated_product_df = gen_df.generate_dataframe(data_list=data_list,column_name_list=column_name_list)
    # logging dataFrame 
    sp_df_logger.log_df(spark_df=generated_product_df,spark_df_name="generated_product_df")
    """Create DataFrame ENDS""" 

    """Join operation STARTS"""
    # initialize df transformation class
    df_t = DataFrameTransformations(spark)
    # initialize df export class
    df_exp = ExportSparkDataFrame(spark)
    # initialize dataframe join classes 
    df_joins = DataFrameJoins(spark)

    #  Handling column ambiguity 
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing inner join opertion on the two dataFrames
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    inner_join_df = df_joins.inner_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    inner_join_df = inner_join_df.drop("prod_id2")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=inner_join_df,spark_df_name="inner_join_df")

    #  Handling column ambiguity 
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing left join operation on the two dataFrames
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    left_join_df = df_joins.left_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    left_join_df = left_join_df.drop("prod_id2")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=left_join_df,spark_df_name="left_join_df")

    #  Handling column ambiguity 
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing right join operation on the two dataframes
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    right_join_df = df_joins.right_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    right_join_df = right_join_df.drop("prod_id2")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=right_join_df,spark_df_name="right_join_df")

    #  Handling column ambiguity 
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing outer join operation on the two dataFrames
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    outer_join_df = df_joins.outer_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    outer_join_df = outer_join_df.drop("prod_id2")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=outer_join_df,spark_df_name="outer_join_df")
    """Join operation ENDS"""
    ######################################
    # CREATE A DATAFRAME AND PERFORM DATAFRAME JOIN ON IT ENDS
    ######################################



    # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
    # input("Please enter")
    spark.stop()
```
# Performance optimization when implementing spark joins
- Spark joins are the one of the most common causes for slowing down a spark application.
- Spark implement two approaches to join your dataFrame 
    - Shuffle join (Shuffle sort merge join) : 
        - This is the most commonly used join type
        - The internal implementation goes back to the notion of the hadoop map-reduce implementation
        - Lets say we have two dataFrames and we want to perform a join operation on the two doesn't matter which join operation.
        - We all know that spark works on driver executor architecture so the two dataframes left and right dataframes will be partitioned and distributed among the executors in the cluster 
        - suppose left dataframe's partition 1 is in executor 1 and right dataframe'1 partition 1 is in executor 2 the join cannot happen until and unless both the left and right dataframe's partition1 are in the same executor. 
        - The join is performed in two stages
            - First stage : (MAP PHASE)
                - In the first stage each executor will map these records using the join key and send it to the exchange
                - These exchanges are called Map Exchange
                - In this first stage all the records are identified by the key and they are made avaliable in the map exchange to be picked up by the spark framework.
                - The map exchange is like a record buffer at the executor.
                - Now the spark framework will pick these records and send them to the reduce exchange and this is where all the magic happens.
                - The reduce exchange will collect all the records for the same key these exchanges are called shuffle partition.
                - The number of shuffle partitions is set by you and it is up to you how many shuffle partitions should be used.
                - The number of shuffle partitions must be equal to the number executors that you are using in the lcuster and this will give you optimal performance 
                - The idea here is that each executor must handle one shuffle partition.
                - All this tranfer of data from map exchange to reduce exchange is called shuffle operation.
                - The shuffle operation can choke your cluster network and significantly slow down your join operation
                - The shuffle is the main reason why the join operation could be slow and not performing.
            - Second stage : ()
                - Now let's assume shuffle is complete : 
                - Now after the shuffle is complete we got our records in the reduce exchange.
                - Each reduce exchange is self sufficient to join the records.
                - Now its all about combining those records and create new dataFrame partitions.
                - Here comes the sort merge join algorithm and finish the job
        - #### **OPTIMIZATION : Tuning join operation for performance:**
            - Tuning your join operation is all about optimizing the shuffle operation.
            - For now since I have this code : 
                ```python
                from typing import List, Tuple, Any
                from lib.logger import Log4j, LogSparkDataframe
                from lib.app_monitor import GetDataFrameMemory
                from pyspark.sql import functions as F

                class GenerateDataFrame:
                    def __init__(self, spark):
                        self.spark = spark
                        self.logger = Log4j(spark)
                        self.sp_df_logger = LogSparkDataframe(spark)
                        self.app_metrics = GetDataFrameMemory(spark)

                    def generate_dataframe(self, data_list : List[Tuple[Any, ...]] = None,column_name_list : List[str]=None):
                        self.logger.debug(f"checking for the supplied data_list: ")
                        if not data_list:
                            self.logger.error("DataList required to generate a dataFrame!")
                            raise ValueError(f"DataList required to generate a dataFrame!")
                        if not column_name_list:
                            self.logger.error("Column List required to generate a dataFrame!")
                            raise ValueError(f"Column List required to generate a dataFrame!")
                        self.logger.debug(f"supplied data_list found : {data_list}")
                        self.logger.debug(f"column name list foind {column_name_list}")

                        # check if the spark session master is set to local if yes then implement repartition if not then don't
                        if self.spark.sparkContext.master == "local[3]":
                            generated_df = (
                                self.spark
                                .createDataFrame(data_list)
                                .toDF(*column_name_list)
                                .repartition(3)
                            )
                        else:
                            generated_df = (
                                self.spark
                                .createDataFrame(data_list)
                                .toDF(*column_name_list)
                            )
                        self.app_metrics.get_mem_usage(generated_df)
                        self.sp_df_logger.log_df_metrics(spark_df=generated_df,spark_df_name="generated_df")
                        return generated_df
                ```
            - The ```generate_dataframe``` function will make sure that the generated dataFrame has three parititions
            - spark.conf
                ```bash
                [SPARK_APP_CONFIGS]
                saprk.app.name = SparkSqlTableDemo
                spark.master = local[3]

                # Setting up the dataset file name that are used in this application
                file_name_csv = flight-time.csv
                file_name_json = flight-time.json
                file_name_parquet = flight-time.parquet
                file_name_text = apache_logs.txt

                # Added a shuffle sort partitions to control the no of partitions of the spark dataFrame in the spark applicaiton
                spark.sql.shuffle.partitions = 2

                # Tell spark to save the created table in this database
                db_name = airline_db
                flight_table_name = flight_data
                ```
            - ```spark.master = local[3]``` in spark.conf will make sure that the spark application uses 3 executors in the cluster for execution the jobs
            - Now I have to make sure that that the shuffle partitions are also set to three 
            - I have a spark.conf file where I needed to add these two lines in order to enforce a specific shuffling paritions
            ```bash
                # Enforce custom shuffle partition count
                spark.sql.shuffle.partitions = 3

                # Disable AQE for deterministic partition behavior
                spark.sql.adaptive.enabled = false
            ```
            - ```spark.sql.shuffle.partitions = 3``` This tells spark that three shuffle partitions must be used but this will only work when I disable the adaptive query engine 
            - ```spark.sql.adaptive.enabled = false``` This will desable adaptive query engine.
            - If you are not performing any action nothing will actually happen since join is a transformation.
            - Here I will be using a dummy action to force the transformation to happen
    - Broadcast join (Broadcast hash join)
        - There two senarios when joining a dataFrame
            - You are trying to join one large dataFrame to another large dataFrame
                - In this you should **watchout** for the following things for the join operation
                    - Don't code like a novice. 
                        - We already know that all the data will be sent from map exchange to reduce exchange during shuffling.
                        - So filtering out the unecessary data beforming performing a join abviously reduces the processing overhead required to move the data from map exchange to reduce exchange 
                        - Sometimes these types of filtering is pretty obvious but some times its not so obvious until and unless you know enough about your dataset.
                        - The rule of thumb is to reduce the size of both left and right dataFrames using filter operation as much as possible before performing join on them.
                        - perform aggregation on the two dataframes even before joining the two dataFrames.
                    - Shuffle partitions and parallelism : 
                        - Look out for the number of shuffle paritions and number of executors
                        - The question you should ask is this what is the maximum number of parallelism I can achieve for my join operation.
                        - The first limit comes from the maximum number of executors you are able to use for your spark application.
                        - The second limit comes from the number of shuffle paritions. Example : If you have configured a 500 executors cluster for your spark application and you are only using 400 shuffle partitions then only 400 executors will be used leaving 100 executors leaving performance on the table hence making the join operation less effiecient.
            - You are trying to join one large dataFrame to a small dataFrame.
        
## NOTICE : Number of partitons of the dataFrame may change when performing outer join.
- In order to understand what is actually happeneing I am attaching the logs from one of my spark application where I was performing joins on two dataFrames
```bash
 logging intermediate dataframe after right join operation
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - Executor memory usage snapshot: {'192.168.1.204:44115': (434.4, 434.31)} MB
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - Memory used by executors (MB): {'192.168.1.204:44115': 0.09} MB
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - result_df :: operation - LogSparkDataframe :: Spark DataFrame Schema (expanded): root
 |-- order_id: string (nullable = true)
 |-- prod_id: string (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- prod_id2: string (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- list_price: long (nullable = true)
 |-- reorder_qty: long (nullable = true)

2025-11-06 14:25:48 DEBUG pyspark-shell:244 - result_df has 3 partitons
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - 
 logging intermediate dataframe after outer join operation
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - Executor memory usage snapshot: {'192.168.1.204:44115': (434.4, 434.35)} MB
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - Memory used by executors (MB): {'192.168.1.204:44115': 0.05} MB
2025-11-06 14:25:48 DEBUG pyspark-shell:244 - result_df :: operation - LogSparkDataframe :: Spark DataFrame Schema (expanded): root
 |-- order_id: string (nullable = true)
 |-- prod_id: string (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- prod_id2: string (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- list_price: long (nullable = true)
 |-- reorder_qty: long (nullable = true)

2025-11-06 14:25:48 DEBUG pyspark-shell:244 - result_df has 2 partitons
```
- I noticed in all the joins except the outer joins the resultant dataFrame has the same number of partitions as when generating the two dataFrames
    - ```python
        from typing import List, Tuple, Any
        from lib.logger import Log4j, LogSparkDataframe
        from lib.app_monitor import GetDataFrameMemory
        from pyspark.sql import functions as F

        class GenerateDataFrame:
            def __init__(self, spark):
                self.spark = spark
                self.logger = Log4j(spark)
                self.sp_df_logger = LogSparkDataframe(spark)
                self.app_metrics = GetDataFrameMemory(spark)

            def generate_dataframe(self, data_list : List[Tuple[Any, ...]] = None,column_name_list : List[str]=None):
                self.logger.debug(f"checking for the supplied data_list: ")
                if not data_list:
                    self.logger.error("DataList required to generate a dataFrame!")
                    raise ValueError(f"DataList required to generate a dataFrame!")
                if not column_name_list:
                    self.logger.error("Column List required to generate a dataFrame!")
                    raise ValueError(f"Column List required to generate a dataFrame!")
                self.logger.debug(f"supplied data_list found : {data_list}")
                self.logger.debug(f"column name list foind {column_name_list}")

                # check if the spark session master is set to local if yes then implement repartition if not then don't
                if self.spark.sparkContext.master == "local[3]":
                    generated_df = (
                        self.spark
                        .createDataFrame(data_list)
                        .toDF(*column_name_list)
                        .repartition(3)
                    )
                else:
                    generated_df = (
                        self.spark
                        .createDataFrame(data_list)
                        .toDF(*column_name_list)
                    )
                self.app_metrics.get_mem_usage(generated_df)
                self.sp_df_logger.log_df_metrics(spark_df=generated_df,spark_df_name="generated_df")
                return generated_df
      ```
- Observation : But when performing the outer join on the two dataFrame the number parition in the resultant dataFrame reduces to 2
- Cause : 
    - Why right join preserves 3 partitions?
        - Spark uses the right DataFrame’s partitioner for the join result (if both sides are hash partitioned on the join key).
        - So if my DataFrame had: ```3 partitions``` Spark will keep 3 partitions after the join. 
        - This is because right join does not require preserving all rows from both sides.
    - Why outer join reduces to 2 partitions?
        - In a full outer join, Spark has to:
            - Preserve all rows from df1
            - Preserve all rows from df2
            - Match keys across both sides
            - Account for null matches on both sides
        - Because of this, Spark does NOT re-use the input partitioner—even when both sides had 3 partitions.
        - nstead, Spark may:
            - Repartition both DataFrames by the join key
            - Use a shuffle exchange
            - Choose an automatic number of partitions
        - Spark auto-calculates shuffle partitions during full outer joins
- Conclusion : What I saw is completely expected and there is nothing wrong with my code.
